In [30]:
import numpy as np
import pandas as pd
import csv
import matplotlib.pyplot as plt
from decimal import Decimal,ROUND_FLOOR
import time
from numba import jit, cuda, njit

%matplotlib inline
%matplotlib notebook

### 1.Load file.

In [2]:
# load the csv file.
path = 'C:/Users/ZWX/PythonNotebooks/UWBM/Unittest/UnsaturatedZone/'
InputData = pd.read_csv(path + 'input_csv.csv')

In [3]:
date = InputData['date']
P_atm = InputData['P_atm']
Ref_grass = InputData['Ref.grass']
E_pot_OW = InputData['E_pot_OW']

### 2. Unsaturated zone ###

In [4]:
# iters = Total timestep.
iters = np.shape(date)[0] 

#### 2.1 General test (Default settings)

##### a. Input data (from other modules): #####
All the input data from other modules are extracted in excel and stored in the specified file in the inputdata folder.

(a). __i_up_uz__

In [5]:
i_up_uz = pd.read_csv(path + 'inputdata/i_up_uz.csv')['i_up_uz_0']

In [6]:
i_up_uz123 = pd.read_csv(path + 'inputdata/i_up_uz_try_nan.csv')['i_up_uz_0']

In [7]:
i_up_uz123[0] # Important to note: If we add squence then the [0] is NA, if not add then [0] is zero, so we may have issue regarding shape

nan

In [8]:
# Very important point to note here: if the first value is set as null instead of zero, then i_up_uz[0] will be 0 other than null, causing the shape of data not match. 
print(np.shape(i_up_uz)) 

(43825,)


(b). __gwl (from groundwater module):__

In [9]:
gwl = pd.read_csv(path + 'inputdata/gwl.csv')['gwl_0']

(c). __meas_uz__

In [10]:
meas_uz = np.zeros(iters)

(d). __Area__

In [11]:
uz_no_meas_area = 6855
uz_meas_area = 0

##### b. ETSelector and SoilSelector####

In [12]:
soilmatrix = pd.read_csv(path + 'soilparameter_new.csv')
etmatrix = pd.read_csv(path + 'ETparameter.csv')

In [13]:
def ETSelector(a, b):
    #a --- soil type, b --- crop type
    
    sol = etmatrix.loc[(etmatrix.soil_type == int(a)) & (etmatrix.crop_type == int(b))]
    return sol

In [14]:
def SoilSelector(a, b, c):
    #a --- soil type, b --- crop type, c --- GWL [m -MSL]
    
    if c>= 0.0 and c <= 2.5:
        c = float(Decimal(str(c)).quantize(Decimal('.1'), rounding=ROUND_FLOOR))
    elif c < 3.0:
        c = 2.5
    elif c < 5.0:
        c = int(c)
    elif c <= 10:
        c = 5.0
    else:
        c = 10.0
    rootzone_thickness = 100 * ETSelector(a, b)['th_rz_m'].values
    sol = soilmatrix.loc[(soilmatrix.soil_type == int(a)) & (soilmatrix.th_rz == int(rootzone_thickness)) & (soilmatrix.gwl == c)]
    return sol

In [15]:
print(list(SoilSelector(2, 1, 1.5)))
SoilSelector(2, 1, 3.3)

['soil_type', 'th_rz', 'gwl', 'moist_cont_eq_rz[mm]', 'capris_max[mm/d]', 'stor_coef', 'k_sat', 'K_unsat']


,soil_type,th_rz,gwl,moist_cont_eq_rz[mm],capris_max[mm/d],stor_coef,k_sat,K_unsat
416,2,40,3.0,163.3,0.083,0.269,6.79,1.22


##### c. Class UnsaturatedZone

In [35]:
class UnsaturatedZone:
    def __init__(self, theta_uz_t0, uz_no_meas_area, uz_meas_area, soiltype = 2, croptype = 1):
        
        # state
        self.init_theta_uz = theta_uz_t0
        
        # parameter
        # uz_meas_area --- unsaturated zone area (with a measure) [m^2].
        # uz_no_meas_area --- unsaturated zone area (without a measure) [m^2].
        # soiltype --- Soil type
        # croptype --- Crop type
        # theta_h3l --- Equilibrium moisture content in rootzone, at which transpiration (Epot ≤ 1 mm/d) reduction starts.
        # theta_h3h --- Equilibrium moisture content in rootzone, at which transpiration (Epot ≥ 5 mm/d) reduction starts.
        # theta_h1 --- Equilibrium moisture content in rootzone with groundwater level at surface level (top rootzone) (complete saturation).
        # theta_h2 --- Equilibrium moisture content in rootzone with groundwater level at bottom rootzone (field capacity).
        # theta_h4 --- Equilibrium moisture content in rootzone, at which transpiration = 0 (wilting point).
        # k_sat_uz --- Predefined saturated permeability of unsaturated zone
        
        self.uz_no_meas_area = uz_no_meas_area
        self.uz_meas_area = uz_meas_area
        self.soiltype = soiltype
        self.croptype = croptype      
    
        et= ETSelector(self.soiltype, self.croptype)
        self.theta_h3l = et['theta_h3l_mm'].values
        self.theta_h3h = et['theta_h3h_mm'].values
        self.theta_h1 = et['theta_h1_mm'].values
        self.theta_h2 = et['theta_h2_mm'].values
        self.theta_h4 = et['theta_h4_mm'].values
        self.k_sat_uz = 10 * SoilSelector(self.soiltype, self.croptype, 1.5)['k_sat'].values # input gwl 1.5 does not affect the K_sat, which is only dependent on soiltype.
    
    def __repr__(self):
        return 'Current P is ' + str(p_atm) + 'Current E is ' + str(e_pot_ow) + '.These are current precipitation and evaporation.'
    #@jit(nogil=True)
    def sol(self, i_up_uz, meas_uz, e_ref, prev_gwl, delta_t = 1 / 24): 
        
        # parameters
        # i_up_uz --- Infiltration from storage on the surface of the unpaved area to the unsaturated zone during the current time step [mm].
        # r_meas_uz --- Inflow from measure area (if applicable) during current time step [mm]
        # theta_h3_uz --- Equilibrium moisture content in the root zone at which reduction of transpiration starts [mm] for the current time step.
        # t_alpha_uz --- Transpiration factor [-] for the current time step.
        # t_atm_uz --- Transpiration from unsaturated zone to atmosphere during the current time step [mm].
        # gwl_up_uz --- First value in predefined table above groundwater level at the end of previous time step [m-SL].
        # gwl_low_uz --- First value in predefined table below groundwater level at the end of previous time step [m-SL].
        # theta_eq_uz --- Equilibrium soil moisture content in the root zone for the current time step [mm].
        # capris_max_uz --- Maximum capillary rise for the current time step [mm/d].
        # theta_uz --- Soil moisture content in the root zone at the end of the current time step [mm]
        
        if self.uz_no_meas_area == 0:
            i_up_uz = r_meas_uz = theta_h3_uz = t_alpha_uz = t_atm_uz = gwl_up_uz = gwl_low_uz = theta_eq_uz = capris_max_uz = p_uz_gw = theta_uz = 0 
        
        else:
            i_up_uz = i_up_uz
            
            r_meas_uz = meas_uz * self.uz_meas_area / self.uz_no_meas_area # May need modifications here.
            
            if e_ref / (2 * delta_t) < 1:
                theta_h3_uz = self.theta_h3l
            elif e_ref / (2 * delta_t) > 5:
                theta_h3_uz = self.theta_h3h
            else:
                theta_h3_uz = self.theta_h3l + (e_ref / (2 * delta_t) - 1) / 4 * (self.theta_h3h - self.theta_h3l)
           
            if self.init_theta_uz + i_up_uz + r_meas_uz > self.theta_h1:
                t_alpha_uz = 0
            elif self.init_theta_uz + i_up_uz + r_meas_uz > self.theta_h2:
                t_alpha_uz = 1 - ((self.init_theta_uz + i_up_uz + r_meas_uz) - self.theta_h2) / (self.theta_h1 - self.theta_h2)
            elif self.init_theta_uz + i_up_uz + r_meas_uz > theta_h3_uz:
                t_alpha_uz = 1
            elif self.init_theta_uz + i_up_uz + r_meas_uz > self.theta_h4:
                t_alpha_uz = ((self.init_theta_uz + i_up_uz + r_meas_uz) - self.theta_h4) / (theta_h3_uz - self.theta_h4)
            else:
                t_alpha_uz = 0
                        
            t_atm_uz = e_ref * t_alpha_uz
            
            c = prev_gwl
            if c>= 0.0 and c <= 2.5:
                c = float(Decimal(str(c)).quantize(Decimal('.1'), rounding=ROUND_FLOOR))
            elif c < 3.0:
                c = 2.5
            elif c < 5.0:
                c = int(c)
            elif c <= 10:
                c = 5.0
            else:
                c = 10.0
            gwl_up_uz = c
            
            if gwl_up_uz < 2.5:
                gwl_low_uz = round(gwl_up_uz + 0.1, 1)
            elif gwl_up_uz < 3:
                gwl_low_uz = 3
            elif gwl_up_uz < 4:
                gwl_low_uz = 4
            elif gwl_up_uz < 5:
                gwl_low_uz = 5
            else:
                gwl_low_uz = 10

            if prev_gwl < 10:
                theta_eq_uz = SoilSelector(self.soiltype, self.croptype, gwl_low_uz)['moist_cont_eq_rz[mm]'].values + (gwl_low_uz - prev_gwl) / (gwl_low_uz - gwl_up_uz) * (SoilSelector(self.soiltype, self.croptype, gwl_up_uz)['moist_cont_eq_rz[mm]'].values - SoilSelector(self.soiltype, self.croptype, gwl_low_uz)['moist_cont_eq_rz[mm]'].values)
            else:
                theta_eq_uz = SoilSelector(self.soiltype, self.croptype, 10)['moist_cont_eq_rz[mm]'].values     
                
            if prev_gwl < 10:
                capris_max_uz = SoilSelector(self.soiltype, self.croptype, gwl_low_uz)['capris_max[mm/d]'].values + (gwl_low_uz - prev_gwl) / (gwl_low_uz - gwl_up_uz) * (SoilSelector(self.soiltype, self.croptype, gwl_up_uz)['capris_max[mm/d]'].values - SoilSelector(self.soiltype, self.croptype, gwl_low_uz)['capris_max[mm/d]'].values)
            else:
                capris_max_uz = SoilSelector(self.soiltype, self.croptype, 10)['capris_max[mm/d]'].values
            
            if self.init_theta_uz + i_up_uz + r_meas_uz - t_atm_uz > theta_eq_uz:
                p_uz_gw = min(self.init_theta_uz + i_up_uz + r_meas_uz - t_atm_uz - theta_eq_uz, delta_t * self.k_sat_uz)
            else:
                p_uz_gw = -1 * min(theta_eq_uz - (self.init_theta_uz + i_up_uz + r_meas_uz - t_atm_uz), delta_t * capris_max_uz)
    
            theta_uz = self.init_theta_uz + i_up_uz + r_meas_uz - t_atm_uz - p_uz_gw
        
            # update state
            self.init_theta_uz = theta_uz

        return i_up_uz, r_meas_uz, theta_h3_uz, t_alpha_uz, t_atm_uz, gwl_up_uz, gwl_low_uz, theta_eq_uz, capris_max_uz, p_uz_gw, theta_uz 

In [17]:
t = 1

I_up_uz = [0]
R_meas_uz = [0]
Theta_h3_uz = [0]
T_alpha_uz = [0] 
T_atm_uz = [0] 
Gwl_up_uz = [0] 
Gwl_low_uz = [0]
Theta_eq_uz = [0] 
Capris_max_uz = [0]
P_uz_gw = [0]

theta_uz_t0 = SoilSelector(2, 1, 1.5)['moist_cont_eq_rz[mm]'].values # 1.5m is initial gwl.
Theta_uz = [theta_uz_t0]

m = UnsaturatedZone(theta_uz_t0, uz_no_meas_area, uz_meas_area, soiltype = 2, croptype = 1)

while t <= iters -1:

    sol = m.sol(i_up_uz[t], meas_uz[t], Ref_grass[t], prev_gwl = gwl[t-1], delta_t = 1/24)
    
    I_up_uz.append(sol[0])
    R_meas_uz.append(sol[1])
    Theta_h3_uz.append(sol[2])
    T_alpha_uz.append(sol[3]) 
    T_atm_uz.append(sol[4]) 
    Gwl_up_uz.append(sol[5])
    Gwl_low_uz.append(sol[6])
    Theta_eq_uz.append(sol[7])
    Capris_max_uz.append(sol[8])
    P_uz_gw.append(sol[9])
    Theta_uz.append(sol[10])

    t += 1
    
filename = 'UZ_General_test_pysol.csv'
np.savetxt('pysol/' + filename, np.c_[I_up_uz, R_meas_uz, Theta_h3_uz, T_alpha_uz, T_atm_uz, Gwl_up_uz, Gwl_low_uz, Theta_eq_uz, Capris_max_uz, P_uz_gw, Theta_uz], fmt = "%.8f", delimiter=',', header = 'I_up_uz, R_meas_uz, Theta_h3_uz, T_alpha_uz, T_atm_uz, Gwl_up_uz, Gwl_low_uz, Theta_eq_uz, Capris_max_uz, P_uz_gw, Theta_uz') 

# Insert the Date column for locating purposes.
df = pd.read_csv('pysol/' + filename)
df.insert(0, 'Date', date)
df.to_csv('pysol/' + filename)

#####  2.1.1 Validation

In [18]:
# read python file
data_py = pd.read_csv('pysol/' + filename)

# read excel file
data_ex = pd.read_csv('exsol/UZ_General_test_exsol.csv')

In [19]:
# Examine (go through all the data)
database = []

for c in range(11):
    for r in range(1,43825): # from row 1 to the last row (row 43824)
        a = data_ex[list(data_ex)[c]][r] - data_py[list(data_py)[c+2]][r]
        database.append(a)
print(max(database))
print(min(database))

1.000000082740371e-07
-1.0000002248489182e-07


#### 2.2 Extended test (Different coefficient sets)

Set 1: soil type = 3, crop type = 1

Set 2: soil type = 7, crop type = 1

Set 3: initial gwl = 3.0

Set 4: initial gwl = 0.0

Set 5: uz_no_meas_area = 0

In [36]:
def validatefunc(a, b, c, d, e, f): # f is the set number.
    t = 1

    I_up_uz = [0]
    R_meas_uz = [0]
    Theta_h3_uz = [0]
    T_alpha_uz = [0] 
    T_atm_uz = [0] 
    Gwl_up_uz = [0] 
    Gwl_low_uz = [0]
    Theta_eq_uz = [0] 
    Capris_max_uz = [0]
    P_uz_gw = [0]

    theta_uz_t0 = SoilSelector(a, b, c)['moist_cont_eq_rz[mm]'].values # c is initial gwl [m].
    Theta_uz = [theta_uz_t0]

    m = UnsaturatedZone(theta_uz_t0, uz_no_meas_area = d, uz_meas_area = e, soiltype = a, croptype = b)

    while t <= iters -1:

        sol = m.sol(i_up_uz[t], meas_uz[t], Ref_grass[t], prev_gwl = gwl[t-1], delta_t = 1/24)
    
        I_up_uz.append(sol[0])
        R_meas_uz.append(sol[1])
        Theta_h3_uz.append(sol[2])
        T_alpha_uz.append(sol[3]) 
        T_atm_uz.append(sol[4]) 
        Gwl_up_uz.append(sol[5])
        Gwl_low_uz.append(sol[6])
        Theta_eq_uz.append(sol[7])
        Capris_max_uz.append(sol[8])
        P_uz_gw.append(sol[9])
        Theta_uz.append(sol[10])

        #print('time step', t)
        t += 1
    filename = 'UZ_General_test_pysol'+ str(f) + '.csv'
    np.savetxt('pysol/' + filename, np.c_[I_up_uz, R_meas_uz, Theta_h3_uz, T_alpha_uz, T_atm_uz, Gwl_up_uz, Gwl_low_uz, Theta_eq_uz, Capris_max_uz, P_uz_gw, Theta_uz], fmt = "%.8f", delimiter=',', header = 'I_up_uz, R_meas_uz, Theta_h3_uz, T_alpha_uz, T_atm_uz, Gwl_up_uz, Gwl_low_uz, Theta_eq_uz, Capris_max_uz, P_uz_gw, Theta_uz') 

    # Insert the Date column for locating purposes.
    df = pd.read_csv('pysol/' + filename)
    df.insert(0, 'Date', date)
    df.to_csv('pysol/' + filename)
    
    data_py = pd.read_csv('pysol/' + filename)
    data_ex = pd.read_csv('exsol/UZ_extended_test_exsol_set'+str(f)+'.csv')
    A = np.zeros((43825, 11))
    for c in range(11):
        for r in range(1,43825):
            A[r,c] = data_ex[list(data_ex)[c]][r] - data_py[list(data_py)[c+2]][r]
    for c in range(11):
        print('COL ' + str(c), 'max', max(A[:,c]), 'min', min(A[:,c]))
        #print(np.where(max(A[:,c]) != 0 and A[:,c] == max(A[:,c])), np.where(min(A[:,c]) != 0 and A[:,c] == min(A[:,c])))
    return 

__soiltype = a, croptype = b, initial_gwl = c, uz_no_meas_area = d, uz_meas_area = e, set number = f__

##### 2.2.1 Set 1: soil type = 3, crop type = 1

In [21]:
start = time.time()
i_up_uz = pd.read_csv(path + 'inputdata/i_up_uz.csv')['i_up_uz_1']
gwl = pd.read_csv(path + 'inputdata/gwl.csv')['gwl_1']
validatefunc(3, 1, 1.5, 6855, 0, 1)
end = time.time()
print(end - start)

COL 0 max 5.000000025123796e-09 min -5.000000025123796e-09
COL 1 max 0.0 min 0.0
COL 2 max 7.000002710810804e-08 min -7.000002710810804e-08
COL 3 max 5.999999941330714e-09 min -6.000000052353016e-09
COL 4 max 5.000000025123796e-09 min -5.000000025123796e-09
COL 5 max 0.0 min 0.0
COL 6 max 0.0 min 0.0
COL 7 max 6.000001917527698e-08 min -6.000001917527698e-08
COL 8 max 6.000000052353016e-09 min -6.000000052353016e-09
COL 9 max 3.9999999978945766e-08 min -3.599999987002889e-08
COL 10 max 9.000001455206075e-08 min -8.000003504093911e-08
529.0274443626404


##### 2.2.2 Set 2: soil type = 7, crop type = 1

In [22]:
i_up_uz = pd.read_csv(path + 'inputdata/i_up_uz.csv')['i_up_uz_2']
gwl = pd.read_csv(path + 'inputdata/gwl.csv')['gwl_2']
validatefunc(7, 1, 1.5, 6855, 0, 2)

COL 0 max 5.000000025123796e-09 min -5.000000025123796e-09
COL 1 max 0.0 min 0.0
COL 2 max 1.3000001075624823e-08 min -1.3999999382008355e-08
COL 3 max 7.999999995789153e-09 min -8.999999967507222e-09
COL 4 max 6.000000052353016e-09 min -6.0000000245974405e-09
COL 5 max 0.0 min 0.0
COL 6 max 0.0 min 0.0
COL 7 max 1.999999810209374e-08 min -1.0000004380117389e-08
COL 8 max 6.000000052353016e-09 min -6.000000052353016e-09
COL 9 max 4.800000000000225e-07 min -3.7670000000000065e-05
COL 10 max 3.000000248221113e-08 min -3.000000248221113e-08


##### 2.2.3 Set 3: initial gwl = 3.0

In [23]:
i_up_uz = pd.read_csv(path + 'inputdata/i_up_uz.csv')['i_up_uz_3']
gwl = pd.read_csv(path + 'inputdata/gwl.csv')['gwl_3']
validatefunc(2, 1, 3, 6855, 0, 3)

COL 0 max 5.000000025123796e-09 min -5.000000025123796e-09
COL 1 max 0.0 min 0.0
COL 2 max 1.000000082740371e-07 min -1.0000002248489182e-07
COL 3 max 5.999999941330714e-09 min -5.999999941330714e-09
COL 4 max 5.000000025123796e-09 min -5.000000025123796e-09
COL 5 max 0.0 min 0.0
COL 6 max 0.0 min 0.0
COL 7 max 6.999997026468918e-08 min -6.00000475969864e-08
COL 8 max 6.000000052353016e-09 min -6.000000052353016e-09
COL 9 max 4.0300000000007e-05 min -3.424999999999956e-05
COL 10 max 9.000001455206075e-08 min -8.000003504093911e-08


##### 2.2.4 Set 4: initial gwl = 0.0

In [37]:
i_up_uz = pd.read_csv(path + 'inputdata/i_up_uz.csv')['i_up_uz_4']
gwl = pd.read_csv(path + 'inputdata/gwl.csv')['gwl_4']
validatefunc(2, 1, 0, 6855, 0, 4)

COL 0 max 5.000000025123796e-09 min -5.000000025123796e-09
COL 1 max 0.0 min 0.0
COL 2 max 1.000000082740371e-07 min -1.0000002248489182e-07
COL 3 max 5.999999941330714e-09 min -5.999999941330714e-09
COL 4 max 5.000000025123796e-09 min -5.000000025123796e-09
COL 5 max 0.0 min 0.0
COL 6 max 0.0 min 0.0
COL 7 max 6.999997026468918e-08 min -6.00000475969864e-08
COL 8 max 7.000000135093387e-09 min -7.000000135093387e-09
COL 9 max 4.148000000000346e-05 min -0.00042564999999999964
COL 10 max 7.000002710810804e-08 min -9.000004297377018e-08


##### 2.2.5 Set 5: uz_no_meas_area = 0

In [25]:
i_up_uz = pd.read_csv(path + 'inputdata/i_up_uz.csv')['i_up_uz_5']
gwl = pd.read_csv(path + 'inputdata/gwl.csv')['gwl_5']
uz_no_meas_area = 0
validatefunc(2, 1, 1.5, 0, 6855, 5)

COL 0 max 0.0 min 0.0
COL 1 max 0.0 min 0.0
COL 2 max 0.0 min 0.0
COL 3 max 0.0 min 0.0
COL 4 max 0.0 min 0.0
COL 5 max 0.0 min 0.0
COL 6 max 0.0 min 0.0
COL 7 max 0.0 min 0.0
COL 8 max 0.0 min 0.0
COL 9 max 0.0 min 0.0
COL 10 max 0.0 min 0.0


Note: when tot_uz_area = 0, the groundwater part is problematic (div 0)